# Task 19: DeepLoc 2.0 Training-Overlap and Generalisation Audit

**Core question:** Is wt_signal_70's performance gain primarily driven by DeepLoc having been trained on the same or highly similar proteins?

Gene-disjoint CV ensures MISFIT train/test genes do not overlap, but cannot prevent:
\[
\text{MISFIT test protein} \in \text{DeepLoc external training set}
\]

DeepLoc 2.0 uses 5-fold CV on Swiss-Prot 2021_04 for the localisation head and a separate sorting-signal training set. Each ensemble member may have seen different subsets of the data. This audit quantifies identifier, sequence, and annotation-level overlap between MISFIT and DeepLoc training sources.

In [ ]:
from pathlib import Path
import hashlib
import json
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

ROOT = Path("/mnt/volume6/czj/labLGN/LabLZ")
BASE = ROOT / "xgboost_trial"
DEEPlOC_DIR = ROOT / "models/deeploc2_package/deeploc2"
DEEPLOC_DATA = ROOT / "models/deeploc2_package/deeploc_data"
SOURCE_CSV = ROOT / "data_preparation/cell2024_model_single_subst.csv"
TASK15_OOF = BASE / "task15_full64_oof.csv"
OOF_18 = BASE / "task18_deeploc_context_oof.csv"
MANIFEST_CSV = DEEPlOC_DIR / "sequence_manifest.csv"
PAIR_MAP_CSV = DEEPlOC_DIR / "variant_pair_map.csv"
UNIPROT_JSON = DEEPlOC_DIR / "uniprot_metadata.json"

OVERLAP_PROTEIN_CSV = DEEPlOC_DIR / "deeploc_overlap_by_protein.csv"
OVERLAP_VARIANT_CSV = DEEPlOC_DIR / "deeploc_overlap_by_variant.csv"
OVERLAP_METRICS_CSV = BASE / "task19_overlap_subgroup_metrics.csv"
OVERLAP_BOOTSTRAP_CSV = BASE / "task19_overlap_bootstrap.csv"

RANDOM_STATE = 42

## 19.1 Identifier-level overlap: UniProt metadata

Query UniProt REST API for all 871 MISFIT UniProt accessions. Classify by review status (Swiss-Prot vs TrEMBL) and presence of subcellular location annotations.

In [ ]:
source = pd.read_csv(SOURCE_CSV)
pairs = pd.read_csv(PAIR_MAP_CSV)
manifest = pd.read_csv(MANIFEST_CSV)
task15 = pd.read_csv(TASK15_OOF)
oof_18 = pd.read_csv(OOF_18)

# Build per-protein table (unique WT)
wt_manifest = manifest[manifest["is_wt"]].copy()
protein_info = source[["Gene", "Uniprot", "sequence"]].drop_duplicates("Uniprot")
assert len(protein_info) == len(wt_manifest) == 871

# Map sequence_id to UniProt via the pairs table
wt_pairs = pairs[["wt_sequence_id", "Gene"]].drop_duplicates("wt_sequence_id")
wt_info = wt_manifest.merge(wt_pairs, left_on="sequence_id", right_on="wt_sequence_id", validate="one_to_one")
wt_info = wt_info.merge(protein_info, on="Gene", validate="one_to_one")
print(f"WT proteins: {len(wt_info)}")
print(f"Unique UniProt: {wt_info['Uniprot'].nunique()}")

In [ ]:
# Load UniProt metadata (pre-queried from REST API)
with open(UNIPROT_JSON) as f:
    uniprot_meta = json.load(f)

# Build overlap table per protein
overlap_protein = wt_info[["sequence_id", "Gene", "Uniprot", "sequence"]].copy()
overlap_protein["uniprot_reviewed"] = overlap_protein["Uniprot"].map(
    lambda a: uniprot_meta.get(a, {}).get("reviewed", False))
overlap_protein["uniprot_has_subcellular_annotation"] = overlap_protein["Uniprot"].map(
    lambda a: len(uniprot_meta.get(a, {}).get("subcellular_locations", [])) > 0)
overlap_protein["uniprot_subcellular_count"] = overlap_protein["Uniprot"].map(
    lambda a: len(uniprot_meta.get(a, {}).get("subcellular_locations", [])))
overlap_protein["in_swissprot"] = overlap_protein["uniprot_reviewed"]

# Summary
print("=== UniProt-level overlap ===")
print(f"Swiss-Prot (reviewed): {overlap_protein['in_swissprot'].sum()}/{len(overlap_protein)}")
print(f"With subcellular annotation: {overlap_protein['uniprot_has_subcellular_annotation'].sum()}/{len(overlap_protein)}")
print(f"Swiss-Prot + subcellular annotation: {(overlap_protein['in_swissprot'] & overlap_protein['uniprot_has_subcellular_annotation']).sum()}")
print(f"Swiss-Prot, no subcellular annotation: {(overlap_protein['in_swissprot'] & ~overlap_protein['uniprot_has_subcellular_annotation']).sum()}")
print(f"TrEMBL (not in Swiss-Prot): {(~overlap_protein['in_swissprot']).sum()}")

# Show the non-Swiss-Prot proteins
non_sp = overlap_protein[~overlap_protein['in_swissprot']]
if len(non_sp):
    print(f"\nNon-Swiss-Prot proteins:")
    print(non_sp[["Gene", "Uniprot", "uniprot_has_subcellular_annotation"]].to_string(index=False))

## 19.2 Sequence-level overlap: SHA256 hashes

Compare MISFIT WT sequences against known sequences. Full sequence hash and Fast/ESM1b effective-sequence hash (aa[:511] + aa[-511:] for proteins >1022 aa).

In [ ]:
def fast_effective_sequence(seq):
    return seq if len(seq) <= 1022 else seq[:511] + seq[-511:]

overlap_protein["sha256_full"] = overlap_protein["sequence"].map(
    lambda s: hashlib.sha256(s.encode()).hexdigest())
overlap_protein["sha256_fast_effective"] = overlap_protein["sequence"].map(
    lambda s: hashlib.sha256(fast_effective_sequence(s).encode()).hexdigest())
overlap_protein["seq_length"] = overlap_protein["sequence"].str.len()
overlap_protein["needs_fast_clipping"] = overlap_protein["seq_length"] > 1022

print(f"Proteins needing Fast clipping (>1022 aa): {overlap_protein['needs_fast_clipping'].sum()}")
print(f"Max sequence length: {overlap_protein['seq_length'].max()}")
print(f"Unique full SHA256: {overlap_protein['sha256_full'].nunique()}")
print(f"Unique Fast-effective SHA256: {overlap_protein['sha256_fast_effective'].nunique()}")

## 19.3 Build overlap categories

Since the exact DeepLoc 2.0 training FASTA files are not bundled with the package, we use a conservative proxy:

- **Category A (likely training overlap):** Swiss-Prot reviewed + has subcellular location annotation → high probability of being in DeepLoc 2.0 localisation training set
- **Category B (possible but uncertain):** Swiss-Prot reviewed, no subcellular location annotation → may have been in training set but less likely
- **Category C (unlikely):** TrEMBL / not in Swiss-Prot → unlikely to be in DeepLoc 2.0 training set

This is a **conservative lower bound** on overlap — actual overlap may be higher if DeepLoc included TrEMBL sequences.

In [ ]:
def classify_overlap(row):
    if row["in_swissprot"] and row["uniprot_has_subcellular_annotation"]:
        return "A_swissprot_with_subcellular"
    elif row["in_swissprot"]:
        return "B_swissprot_no_subcellular"
    else:
        return "C_trembl_or_other"

overlap_protein["overlap_category"] = overlap_protein.apply(classify_overlap, axis=1)

print("=== Overlap categories (per unique WT protein) ===")
cat_counts = overlap_protein["overlap_category"].value_counts()
for cat in ["A_swissprot_with_subcellular", "B_swissprot_no_subcellular", "C_trembl_or_other"]:
    count = cat_counts.get(cat, 0)
    print(f"  {cat}: {count} proteins")

# These are the boolean flags for the variant-level table
overlap_protein["likely_in_deeploc_training"] = overlap_protein["overlap_category"] == "A_swissprot_with_subcellular"
overlap_protein.to_csv(OVERLAP_PROTEIN_CSV, index=False)
print(f"\nSaved: {OVERLAP_PROTEIN_CSV}")

## 19.4 Map overlap to variant level

Propagate per-protein overlap flags to all 2,179 variants for stratified evaluation.

In [ ]:
# Map via Gene (one protein per gene in MISFIT)
gene_overlap = overlap_protein[["Gene", "overlap_category", "likely_in_deeploc_training",
    "in_swissprot", "uniprot_has_subcellular_annotation", "needs_fast_clipping"]].copy()

overlap_variant = pairs[["KEY", "Gene", "wt_sequence_id", "Mislocalized"]].merge(
    gene_overlap, on="Gene", validate="many_to_one")
assert len(overlap_variant) == 2179

overlap_variant.to_csv(OVERLAP_VARIANT_CSV, index=False)
print(f"Variants: {len(overlap_variant)}")
print(f"Saved: {OVERLAP_VARIANT_CSV}")

# Merge with OOF predictions for stratified evaluation
eval_df = overlap_variant.merge(oof_18[["KEY", "oof_xgboost_64", "oof_wt_signal_70"]], on="KEY", validate="one_to_one")
eval_df = eval_df.merge(task15[["KEY", "final_alphamissense_score", "fold"]], on="KEY", how="left", validate="one_to_one")
print(f"\nCategory distribution (variants):")
for cat in ["A_swissprot_with_subcellular", "B_swissprot_no_subcellular", "C_trembl_or_other"]:
    sub = eval_df[eval_df["overlap_category"] == cat]
    if len(sub):
        print(f"  {cat}: n={len(sub)}, positives={int(sub['Mislocalized'].sum())} (prevalence={sub['Mislocalized'].mean():.3f})")
    else:
        print(f"  {cat}: n=0")

## 19.5 Stratified evaluation: wt_signal_70 performance by overlap category

For each overlap subgroup, report 64D and 70D metrics, the paired difference, and note whether the subgroup has enough positives for reliable estimation.

In [ ]:
def subgroup_metrics(eval_df, subgroup_name, subgroup_mask):
    sub = eval_df[subgroup_mask]
    y_sub = sub["Mislocalized"].astype(int).to_numpy()
    n_pos = int(y_sub.sum())
    prevalence = y_sub.mean()

    if n_pos < 5:
        return {"subgroup": subgroup_name, "n_variants": len(sub), "n_positives": n_pos,
                "prevalence": prevalence, "warning": "too few positives for reliable estimation"}

    s64 = sub["oof_xgboost_64"].to_numpy()
    s70 = sub["oof_wt_signal_70"].to_numpy()

    am_mask = sub["final_alphamissense_score"].notna().to_numpy()
    s_am = sub.loc[am_mask, "final_alphamissense_score"].to_numpy() if am_mask.sum() > 0 else None

    return {
        "subgroup": subgroup_name,
        "n_variants": len(sub),
        "n_genes": sub["Gene"].nunique(),
        "n_positives": n_pos,
        "prevalence": prevalence,
        "auroc_64": roc_auc_score(y_sub, s64),
        "auroc_70": roc_auc_score(y_sub, s70),
        "auprc_64": average_precision_score(y_sub, s64),
        "auprc_70": average_precision_score(y_sub, s70),
        "delta_auroc": roc_auc_score(y_sub, s70) - roc_auc_score(y_sub, s64),
        "delta_auprc": average_precision_score(y_sub, s70) - average_precision_score(y_sub, s64),
        "am_paired_n": int(am_mask.sum()) if am_mask.sum() > 0 else 0,
        "auroc_am": roc_auc_score(y_sub[am_mask], s_am) if s_am is not None else None,
        "auprc_am": average_precision_score(y_sub[am_mask], s_am) if s_am is not None else None,
    }

rows = []
rows.append(subgroup_metrics(eval_df, "All variants", np.ones(len(eval_df), dtype=bool)))
for cat, label in [("A_swissprot_with_subcellular", "A: Swiss-Prot + subcellular annotation"),
                    ("B_swissprot_no_subcellular", "B: Swiss-Prot, no subcellular annotation"),
                    ("C_trembl_or_other", "C: TrEMBL / other")]:
    mask = eval_df["overlap_category"] == cat
    if mask.sum() > 0:
        rows.append(subgroup_metrics(eval_df, label, mask))

# Combine A+B as "any Swiss-Prot"
rows.append(subgroup_metrics(eval_df, "A+B: Any Swiss-Prot",
    eval_df["overlap_category"].isin(["A_swissprot_with_subcellular", "B_swissprot_no_subcellular"])))

metrics_df = pd.DataFrame(rows)
print(metrics_df.to_string(index=False))
metrics_df.to_csv(OVERLAP_METRICS_CSV, index=False)
print(f"\nSaved: {OVERLAP_METRICS_CSV}")

## 19.6 Gene-cluster bootstrap: ΔAUPRC stratified by overlap

Compute the difference-in-differences: (70D−64D improvement in overlap subgroup) minus (70D−64D improvement in non-overlap subgroup).

This directly tests whether the wt_signal_70 increment differs by overlap status.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

# Only meaningful if both subgroups have enough data
cat_a_mask = (eval_df["overlap_category"] == "A_swissprot_with_subcellular").to_numpy()
cat_bc_mask = ~cat_a_mask  # B + C combined (proteins without subcellular annotation or not in Swiss-Prot)

n_a_pos = int(eval_df.loc[cat_a_mask, "Mislocalized"].sum())
n_bc_pos = int(eval_df.loc[cat_bc_mask, "Mislocalized"].sum())
print(f"Category A (with subcellular): {cat_a_mask.sum()} variants, {n_a_pos} positives")
print(f"Category B+C (without/other): {cat_bc_mask.sum()} variants, {n_bc_pos} positives")

y_all = eval_df["Mislocalized"].astype(int).to_numpy()
s64 = eval_df["oof_xgboost_64"].to_numpy()
s70 = eval_df["oof_wt_signal_70"].to_numpy()

if n_a_pos >= 10 and n_bc_pos >= 5:
    rng = np.random.default_rng(RANDOM_STATE)
    unique_genes = eval_df["Gene"].astype(str).unique()
    gene_to_idx = {g: np.flatnonzero(eval_df["Gene"].astype(str).to_numpy() == g) for g in unique_genes}

    overlap_bootstrap_rows = []
    for rep in range(2000):
        sampled = rng.choice(unique_genes, size=len(unique_genes), replace=True)
        idx = np.concatenate([gene_to_idx[g] for g in sampled])
        if np.unique(y_all[idx]).size < 2:
            continue
        # Subgroup masks within bootstrap sample
        a_idx = idx[cat_a_mask[idx]]
        bc_idx = idx[cat_bc_mask[idx]]
        
        metrics = {"replicate": rep}
        for subgroup_name, sub_idx in [("A", a_idx), ("BC", bc_idx), ("all", idx)]:
            if len(np.unique(y_all[sub_idx])) < 2:
                continue
            d_auc = roc_auc_score(y_all[sub_idx], s70[sub_idx]) - roc_auc_score(y_all[sub_idx], s64[sub_idx])
            d_ap = average_precision_score(y_all[sub_idx], s70[sub_idx]) - average_precision_score(y_all[sub_idx], s64[sub_idx])
            metrics[f"{subgroup}_delta_auroc"] = d_auc
            metrics[f"{subgroup}_delta_auprc"] = d_ap
        
        # Difference-in-differences
        if "A_delta_auprc" in metrics and "BC_delta_auprc" in metrics:
            metrics["did_delta_auroc"] = metrics["A_delta_auroc"] - metrics["BC_delta_auroc"]
            metrics["did_delta_auprc"] = metrics["A_delta_auprc"] - metrics["BC_delta_auprc"]
        overlap_bootstrap_rows.append(metrics)

    overlap_bootstrap = pd.DataFrame(overlap_bootstrap_rows)
    overlap_bootstrap.to_csv(OVERLAP_BOOTSTRAP_CSV, index=False)

    print("\n=== Subgroup ΔAUPRC (70D − 64D) ===")
    for subgroup in ["A", "BC", "all"]:
        col = f"{subgroup}_delta_auprc"
        if col in overlap_bootstrap.columns and overlap_bootstrap[col].notna().sum() > 100:
            vals = overlap_bootstrap[col].dropna()
            print(f"  {subgroup}: mean={vals.mean():+.4f}, 95% CI [{vals.quantile(0.025):+.4f}, {vals.quantile(0.975):+.4f}]")

    if "did_delta_auprc" in overlap_bootstrap.columns:
        did = overlap_bootstrap["did_delta_auprc"].dropna()
        print(f"\n  Difference-in-differences (A − BC): mean={did.mean():+.4f}, 95% CI [{did.quantile(0.025):+.4f}, {did.quantile(0.975):+.4f}]")
else:
    print("Insufficient positives for stratified bootstrap.")

## 19.7 Composition audit

Check whether overlap subgroups differ systematically in protein class, positive prevalence, or other confounders.

In [ ]:
# Merge with source metadata
comp_df = eval_df.merge(source[["Gene", "wt_primary"]].drop_duplicates("Gene"), on="Gene", validate="many_to_one")

print("=== Primary localisation distribution by overlap category ===")
for cat in ["A_swissprot_with_subcellular", "B_swissprot_no_subcellular", "C_trembl_or_other"]:
    sub = comp_df[comp_df["overlap_category"] == cat]
    if len(sub) == 0:
        continue
    print(f"\n{cat} (n={len(sub)}, prevalence={sub['Mislocalized'].mean():.3f}):")
    loc_dist = sub["wt_primary"].value_counts()
    for loc, count in loc_dist.head(5).items():
        print(f"    {loc}: {count} ({count/len(sub)*100:.1f}%)")

print("\n=== Positive prevalence by overlap category ===")
print(comp_df.groupby("overlap_category")["Mislocalized"].agg(["count", "sum", "mean"]).to_string())

## 19.8 Summary and interpretation

The overlap audit does not invalidate wt_signal_70's performance — it characterises **what kind of generalisation** the model
achieves. The results determine whether the improvement represents generalisable signal for unseen proteins, external supervised
protein knowledge, or a combination of both.

**Key outputs:**
- `deeploc_overlap_by_protein.csv` — overlap flags for each of the 871 unique WT proteins
- `deeploc_overlap_by_variant.csv` — overlap flags propagated to 2,179 variants
- `task19_overlap_subgroup_metrics.csv` — stratified AUROC/AUPRC by overlap category
- `task19_overlap_bootstrap.csv` — paired gene-cluster bootstrap with difference-in-differences